In [7]:
import json
import numpy as np
import pandas as pd
import sympy as sp

In [2]:
with open('netlist.json', 'r') as f:
    netlist_data = json.load(f) 

In [4]:
components_df = pd.DataFrame(netlist_data['components'])
passive_comp = ['resistor', 'capacitor', 'inductor']
source_comp = ['voltage-source', 'current-source']
passive_comp_df = components_df[components_df['type'].isin(passive_comp)].copy()
source_df = components_df[components_df['type'].isin(source_comp)].copy()
passive_comp_df.reset_index(drop=True, inplace=True)
source_df.reset_index(drop=True, inplace=True)

# add source definitions to columns

source_df['source_type'] = source_df['source_definition'].apply(
    lambda sd: sd.get('type') if isinstance(sd, dict) else None
)
source_df['phase_angle'] = source_df['source_definition'].apply(
    lambda sd: sd.get('phase_angle', 0) if isinstance(sd, dict) else 0
)
source_df['frequency'] = source_df['source_definition'].apply(
    lambda sd: sd.get('frequency') if isinstance(sd, dict) else None
)
source_df = source_df.drop(columns = ['source_definition'])
passive_comp_df = passive_comp_df.drop(columns = ['source_definition'])
print(passive_comp_df)
print(source_df)

        type  node1  node2 value
0  capacitor      1      2    10
1   resistor      2      3     2
2   inductor      0      4     5
3   resistor      0      2    20
4  capacitor      3      4    10
             type  node1  node2 value source_type  phase_angle  frequency
0  voltage-source      0      1    10          ac            0      100.0
1  current-source      4      3   100          dc            0        NaN


In [9]:
tot_nodes = 4 #exlcuding ground
s = sp.symbols('s')

In [16]:
#G matrix

G = sp.Matrix.zeros(tot_nodes, tot_nodes)
for _, comp in components_df.iterrows():
    type = comp['type']
    node_1 = comp['node1']
    node_2 = comp['node2']
    value = float(comp['value'])
    
    #admittance
    if type == 'resistor':
        y = 1/value
    elif type == 'capacitor':
        y = s * value
    elif type == 'inductor':
        y = 1/(s*value)
    else:
        continue
    
    #matrix input
    if node_1 == 0:
        G[node_2-1, node_2-1] += y
    if node_1 != 0:
        G[node_1-1, node_1-1] += y
    if node_1 != 0 and node_2 !=0:
        G[node_1-1, node_2-1] -= y
        G[node_2-1, node_1-1] -= y
    else:
        continue
       
    
print(G)   

Matrix([[10.0*s, -10.0*s, 0, 0], [-10.0*s, 0.550000000000000, -0.500000000000000, 0], [0, -0.500000000000000, 10.0*s, -10.0*s], [0, 0, -10.0*s, 0.2/s]])
